# NORMA quickstart

Discover constraints, surface **errors vs rare-but-valid** values, and export to your stack.

```bash
pip install norma-dq
```

In [ ]:
import pandas as pd
from norma.core.table import Table

# a tiny dirty table: country -> continent holds, except one corrupted cell
df = pd.DataFrame({
    'country':   ['Japan']*5 + ['Kenya']*5,
    'continent': ['Asia']*4 + ['Europe'] + ['Africa']*5,   # 'Europe' breaks country -> continent
})
t = Table.from_pandas(df)
report = t.profile()   # renders an HTML report inline (colour-coded violations)
report

In [ ]:
# the discovered constraints, in text form
print(report.to_text())

In [ ]:
# export the rules to Great Expectations (or pandera / dbt / sql_check / ...)
from norma.export import export
files = export(report, 'great_expectations')
print(list(files))
print(files['norma_suite.json'][:400])

In [ ]:
# freeze a versioned contract and re-validate (this is what `norma check` does in CI)
from norma import contract as ct
contract = ct.freeze(report)
for r in ct.check(t, contract):
    print('ok ' if r.ok else 'FAIL', r.id, '-', r.detail)